
# Corpus Summary Statistics

Load the plain text files from `processed/` to compute line and token counts
for each corpus used in the stylometry analysis. Outputs a LaTeX table with
columns: Author, Works, Lines, Tokens.

The cento itself is shown separately at the bottom.


In [ ]:

from pathlib import Path
import pandas as pd

PROCESSED = Path("processed")

def count_lines_and_tokens(text_path):
    """Count hexameter lines and word tokens in a plain text file."""
    text = text_path.read_text(encoding="utf-8")
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    tokens = sum(len(line.split()) for line in lines)
    return len(lines), tokens

def collect_corpus_stats(subdir):
    """Collect line/token stats for all .txt files (not sentence files) in a directory."""
    records = []
    for path in sorted(PROCESSED.glob(f"{subdir}/*.txt")):
        # Skip sentence/masked files
        if "_sentences" in path.name:
            continue
        lines, tokens = count_lines_and_tokens(path)
        records.append({"file": path.stem, "lines": lines, "tokens": tokens})
    return pd.DataFrame(records)



## Collect Statistics


In [ ]:

# Collect stats for each corpus
ausonius_df = collect_corpus_stats("ausonius")
vergil_df = collect_corpus_stats("vergil")
aeneid_df = collect_corpus_stats("aeneid")
background_df = collect_corpus_stats("background")
cento_df = collect_corpus_stats("question")

# Build the table rows
rows = []

# Vergil — Georgics
rows.append({
    "Author": "Vergil",
    "Works": r"\emph{Georgics}",
    "Lines": int(vergil_df["lines"].sum()),
    "Tokens": int(vergil_df["tokens"].sum()),
})

# Vergil — Aeneid
rows.append({
    "Author": "Vergil",
    "Works": r"\emph{Aeneid}",
    "Lines": int(aeneid_df["lines"].sum()),
    "Tokens": int(aeneid_df["tokens"].sum()),
})

# Ausonius (excludes cento)
rows.append({
    "Author": "Ausonius",
    "Works": r"all stichic hexameter",
    "Lines": int(ausonius_df["lines"].sum()),
    "Tokens": int(ausonius_df["tokens"].sum()),
})

# Cento Nuptialis (separate row, attributed to Ausonius)
rows.append({
    "Author": "Ausonius",
    "Works": r"\emph{Cento Nuptialis}",
    "Lines": int(cento_df["lines"].sum()),
    "Tokens": int(cento_df["tokens"].sum()),
})

# Background authors — each has a single work
BG_WORKS = {
    "lucan": r"\emph{Bellum Civile}",
    "ovid": r"\emph{Metamorphoses}",
    "silius": r"\emph{Punica}",
    "valerius": r"\emph{Argonautica}",
}

bg_authors = {}
for _, row in background_df.iterrows():
    parts = row["file"].split("_", 1)
    author_key = parts[0].lower()
    if author_key == "silius":
        author = "Silius Italicus"
    elif author_key == "valerius":
        author = "Valerius Flaccus"
    else:
        author = parts[0].capitalize()
    bg_authors.setdefault(author, {"key": author_key, "rows": []})["rows"].append(row)

for author in sorted(bg_authors):
    info = bg_authors[author]
    author_rows = info["rows"]
    work_name = BG_WORKS.get(info["key"], info["key"].capitalize())
    total_lines = sum(r["lines"] for r in author_rows)
    total_tokens = sum(r["tokens"] for r in author_rows)
    rows.append({
        "Author": author,
        "Works": work_name,
        "Lines": int(total_lines),
        "Tokens": int(total_tokens),
    })

df = pd.DataFrame(rows)
display(df)



## LaTeX Output

Raw `pandas.DataFrame.to_latex()` output for pasting into the paper.


In [ ]:

print(df.to_latex(index=False, escape=False))
